# 3.1 Demo - Spark UI Introduction
**Course:** Databricks Performance Optimization · **Lesson:** Spark UI Introduction

Every performance investigation starts in the Spark UI. Learn the hierarchy first:

**Query → Job(s) → Stage(s) → Task(s)**
- An **action** (`count`, `write`, `collect`, `display`) triggers one or more **jobs**
- A job is split into **stages** at every **shuffle** boundary (wide transformation)
- A stage runs one **task per partition**. Tasks are the unit of parallelism (one task per core at a time)

On Databricks: *Compute → your cluster → Spark UI*, or the *View* link under a cell. Locally: open the URL printed by the setup cell (default http://127.0.0.1:24040).

![An action starts a job; the job is split into stages at the shuffle; each stage runs one task per partition](../images/3.1-query-jobs-stages-tasks.svg)

**How to read it:** the grey boxes live on the driver, the green ones are executor work.
Section A below produces only the left stage (8 tasks, one per partition). Section B adds a `groupBy`, which
inserts the shuffle box and a second stage: that is the pair of stages you will see in `stage_summary`.

In [ ]:
from adelab.classroom import setup
from adelab import perf
from pyspark.sql import functions as F

DA = setup("3.1", reset=True)
print("Open the Spark UI:", spark.sparkContext.uiWebUrl)

## A. A narrow-only query = 1 stage
`filter` and `withColumn` are **narrow**: each output partition depends on one input partition, so there is no shuffle.

In [ ]:
df = spark.range(0, 5_000_000, numPartitions=8).withColumn("bucket", F.col("id") % 100)
spark.sparkContext.setJobDescription("A: narrow only")
print(df.filter("bucket = 7").count())

**In the UI → Jobs tab**: find job *"A: narrow only"*. It has one stage with 8 tasks. Open the stage. The **Event Timeline** shows 8 tasks running in parallel across your cores.

## B. A wide transformation = a shuffle = a new stage

![Narrow: each output partition depends on one input partition. Wide: every output partition reads from every input partition through a shuffle](../images/3.1-narrow-vs-wide.svg)

**How to read it:** on the left, `filter` keeps each partition where it is, so section A stays in one stage.
On the right, `groupBy("bucket")` must bring all rows of a key together, so every input partition sends rows to
every output partition. That all-to-all exchange is the *Shuffle Write* / *Shuffle Read* you see in the next cell.

In [ ]:
spark.sparkContext.setJobDescription("B: groupBy (wide)")
result = df.groupBy("bucket").agg(F.sum("id").alias("total"))
result.write.format("noop").mode("overwrite").save()
display(perf.stage_summary(spark, last=3))

In **Stages**, the first stage has *Shuffle Write*, the second has *Shuffle Read*. The second stage has fewer tasks than `spark.sql.shuffle.partitions`, because **AQE** coalesced small partitions.

In [ ]:
print("spark.sql.shuffle.partitions =", spark.conf.get("spark.sql.shuffle.partitions"))
print("AQE enabled                  =", spark.conf.get("spark.sql.adaptive.enabled"))

## C. The SQL / DataFrame tab: the query plan with metrics
This is the most useful tab. It shows the **physical plan** with runtime metrics on every node: rows, files read, bytes, shuffle size, spill, time. `perf.run()` runs a query and reads the same numbers through the UI's REST API:

In [ ]:
display(perf.compare(perf.run(result, "groupBy bucket")))

In [ ]:
result.explain(mode="formatted")

## D. Storage tab: caching
`cache()` is lazy. The data shows up in the **Storage** tab after the first action. Remember to `unpersist()`, because cached data uses executor memory that joins and aggregations also need.

In [ ]:
cached = df.filter("bucket < 10").cache()
cached.count()
print("cached:", cached.is_cached)
cached.unpersist()

## E. Environment and Executors tabs
- **Environment**: every effective Spark config. Check here whether your setting really applied.
- **Executors**: cores, memory, GC time, failed tasks, shuffle and spill totals per executor. Locally there is one driver that is also the executor.

## F. What to look for (checklist used in the next lessons)
| Symptom in the UI | Likely cause | Lesson |
|---|---|---|
| Thousands of tiny tasks, long *scheduler delay* | Small files / too many partitions | 3.2 File Explosion |
| *Files read* ≈ all files for a selective filter | No data skipping (bad layout) | 3.3 Data Skipping & Liquid Clustering |
| Large *Shuffle Write / Read* | Wide transformations, missing broadcast | 3.4 Shuffle |
| One task 10x longer than the median | **Skew** | 3.5 Exploding Join |
| *Spill (memory/disk)* > 0 | Partitions too large for execution memory | 3.5 Exploding Join |
| Long *Python* time / `BatchEvalPython` in the plan | Row-at-a-time Python UDFs (serialization) | 3.6 UDFs |

In [ ]:
DA.cleanup()